# 1) Project Setup

This section defines imports, random seed, and file paths used throughout the notebook.

In [2]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Reproucibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Display settings
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

# Project paths
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data"
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"

print("Project root:", PROJECT_ROOT)
print("Data folder exists:", DATA_DIR.exists())
print("train.csv exists:", TRAIN_PATH.exists())
print("test.csv exists:", TEST_PATH.exists())

Project root: c:\Users\Xslayer77\Documents\kaggle_titanic
Data folder exists: True
train.csv exists: True
test.csv exists: True


# 2) Load Data

Purpose: Load training and test datasets into pandas DataFrames.

In this section I will:
- Read the CSV files for train and test data.
- Print dataset shapes to verify expected row counts.
- Preview first rows to confirm successful loading.

Expected outcome:
- Train and test DataFrames are available.
- Basic structure of each dataset is visible.

In [3]:
# Load data
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

display(train_df.head())
display(test_df.head())

Train shape: (891, 12)
Test shape: (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


# 3) Data Quality Checks

Purpose: Understand data completeness and structure before modeling.

In this section I will:
- Check missing values by column.
- Inspect data types for each feature.
- Review target distribution for Survived.

Expected outcome:
- Clear list of columns needing cleaning or imputation.
- Awareness of class balance in the target.

In [5]:
# Data Quality Checks

# Missing values
missing_train = train_df.isnull().sum().sort_values(ascending=False)
missing_test = test_df.isnull().sum().sort_values(ascending=False)

print("Missing values in train (top 10):")
display(missing_train.head(10).to_frame("missing_count"))

print("Missing values in test (top 10):")
display(missing_test.head(10).to_frame("missing_count"))

# Data types
print("Train dtypes:")
display(train_df.dtypes.to_frame("dtype"))

print("Test dtypes:")
display(test_df.dtypes.to_frame("dtype"))

# Target distribution
target_counts = train_df["Survived"].value_counts().sort_index()
target_ratio = train_df["Survived"].value_counts(normalize=True).sort_index()

print("Target counts (0=Did not survive, 1=Survived):")
display(target_counts.to_frame("count"))

print("Target proportions:")
display((target_ratio * 100).round(2).to_frame("percent"))

Missing values in train (top 10):


,missing_count
Cabin,687
Age,177
Embarked,2
PassengerId,0
Name,0
Pclass,0
Survived,0
Sex,0
Parch,0
SibSp,0


Missing values in test (top 10):


,missing_count
Cabin,327
Age,86
Fare,1
Name,0
Pclass,0
PassengerId,0
Sex,0
Parch,0
SibSp,0
Ticket,0


Train dtypes:


...

,dtype
PassengerId,int64
Pclass,int64
Name,str
Sex,str
Age,float64
SibSp,int64
Parch,int64
Ticket,str
Fare,float64
Cabin,str


Target counts (0=Did not survive, 1=Survived):


,count
Survived,
0,549
1,342


Target proportions:


,percent
Survived,
0,61.62
1,38.38


### Unique Values per Column

This check shows how many distinct values each column has (including missing values), which helps identify low-cardinality categorical features and high-cardinality fields like Name or Ticket.

In [6]:
# Unique values per column (including NaN)
unique_train = train_df.nunique(dropna=False).sort_values(ascending=False)
unique_test = test_df.nunique(dropna=False).sort_values(ascending=False)

print("Unique values in train:")
display(unique_train.to_frame("unique_count"))

print("Unique values in test:")
display(unique_test.to_frame("unique_count"))

Unique values in train:


,unique_count
PassengerId,891
Name,891
Ticket,681
Fare,248
Cabin,148
Age,89
SibSp,7
Parch,7
Embarked,4
Pclass,3


Unique values in test:


,unique_count
PassengerId,418
Name,418
Ticket,363
Fare,170
Age,80
Cabin,77
Parch,8
SibSp,7
Pclass,3
Embarked,3


## Observation
- **Cabin** has a large number of missing values.
- **Age** and **Embarked** require imputation
- **PassengerId**, **Name** and **Ticket** has too many unique values to be of use
- **Name** may be useful for feature extraction (e.g., titles)